# Demo: Snekmer Learn/Apply


 <b>Learn/Apply</b> is a protein annotation method that uses cosine similarity to compare a user-generated kmer counts matrix to the kmer counts of an novel genome, predicting the annotation of each protein within.

**Learn** relies on a large training set of genomes to make predictions. With each addition to the training set, the accuracy increases.

**Apply** requires several outputs from Learn as well some novel genomes or sequences.


In this notebook, we will demonstrate how to use Snekmer Learn/Apply with small training dataset of 20 genomes and 2 "unknown" genomes.



## Getting Started with Snekmer Learn

### Setup

First, install Snekmer using the instructions in the [user installation guide](https://github.com/PNNL-CompBio/Snekmer/).

Before running Snekmer, verify that files have been placed in an **_input_** directory placed at the same level as the **_config.yaml_** file. The assumed file directory structure is illustrated below.

    .
    ├── input
    │   ├── A.fasta
    │   ├── B.fasta
    │   ├── C.fasta
    │   ├── D.fasta
    │   └── etc.
    ├── config.yaml
    ├── annotations
        └── annotations.ann
    
(Note: Snekmer automatically creates the **_output_** directory when creating output files, so there is no need to create this folder in advance.)

To ensure that snekmer is available in the Jupyter notebook do the following:
    `conda activate snekmer`
    `conda install -c anaconda ipykernel`
    `python -m ipykernel install --user --name=snekmer`
    `jupyter notebook`







### Notes on Using Snekmer

Snekmer assumes that the user will primarily process input files using the command line. For more detailed instructions, refer to the [README](https://github.com/PNNL-CompBio/Snekmer).

The basic process for running Snekmer Learn/Apply is as follows:

1. Verify that your file directory structure is correct and that the top-level directory contains a **_config.yaml_** file.
    - A **_config.yaml_** template has been included in the Snekmer codebase at **_resources/learn_apply/config.yaml_**.
2. Modify the **_config.yaml_** with the desired parameters.
3. Use the command line to navigate to the directory containing both the **_config.yaml_** file and **_input_** directory.
4. Run `snekmer learn`, then copy the appropriate outputs to a seperate directory to run `snekmer apply`


## Running Snekmer Learn Pipeline

### Setup

To set up the workflow such that operation mimics the command line implementation of Snekmer Learn/Apply, we will initialize a dictionary (rather than a YAML file) and gather all input files. Input files are detected here using `glob.glob`, exactly as Snekmer performs input file detection.

In [1]:
# built-in imports
import csv
import itertools
import os
import pickle
import re
import shutil
import sys
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as csv
import sklearn
import snekmer as skm
from Bio import SeqIO

### Configuration File Input

In [2]:
# define config
# (note: handled via config.yaml in the snekmer CLI workflow)

config = {
    
    # required parameters
    "k": 8,
    "alphabet": 2,  # choices 0-5 or names (see alphabet module), or None
    "min_rep_thresh": 1,
    "processes": 2,

    # input handling
    "input": {
        "example_index_file": False,
        "feature_set": False,
        "file_extensions": ["fasta", "fna", "faa", "fa"],
        "regex": r"[a-z]{3}[A-Z]{1}",  # regex to parse family from filename
    },
    
    # output handling
    "output": {
        "nested_dir": False,  # if True, saves into {save_dir}/{alphabet name}/{k}
        "verbose": True, # if True, logs verbose outputs
        "format": "simple",  # choices: ["simple", "gist", "sieve"]
        "filter_duplicates": True,
        "n_terminal_file": False,
        "shuffle_n": False,
        "shuffle_sequences": False,
    },
    
    # LearnApply Parameters
    "learnapp": {
        "save_apply_associations": False,
        "selection": "top_hit",
        "threshold_type": "Median"
    }

}

### Rule 0: Receive files

Before going through the workflow, we glob all filenames contained within the input directory that end in the pre-defined file extensions and/or the extension and `.gz`. These files should be FASTA format sequence files or the gzip-compressed versions of such files.

Note that while in this notebook, the path to the demo files is specified with the `input_dir` variable, the Snekmer CLI assumes that input files are stored according to the file structure specified above in the **Setup** section.

In [3]:
# collect all fasta-like files, unzipped filenames, and basenames
input_dir = "learnapp_tutorial_files/learn/input/"
input_files = glob(os.path.join(input_dir, "*"))
zipped = [fa for fa in input_files if fa.endswith(".gz")]
unzipped = [
    fa.rstrip(".gz")
    for fa, ext in itertools.product(input_files, config["input"]["file_extensions"])
    if fa.rstrip(".gz").endswith(f".{ext}")
]

print("zipped files:\t", zipped)
print("unzipped files:\t", unzipped)

zipped files:	 []
unzipped files:	 ['learnapp_tutorial_files/learn/input/uniprot_reduced.fasta']


In [4]:
# map extensions to basename (basename.ext.gz -> {basename: ext})
UZ_MAP = {
    skm.utils.split_file_ext(f)[0]: skm.utils.split_file_ext(f)[1] for f in zipped
}

FA_MAP = {
    skm.utils.split_file_ext(f)[0]: skm.utils.split_file_ext(f)[1] for f in unzipped
}

UZS = list(UZ_MAP.keys())
FAS = list(FA_MAP.keys())

print("zipped filename wildcards:\t", UZS)
print("unzipped filename wildcards:\t", FAS)

zipped filename wildcards:	 []
unzipped filename wildcards:	 ['uniprot_reduced']


In [5]:
# define output directory (and create if missing)
output_dir = "learnapp_tutorial_files/learn/output"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print("output directory:\t", output_dir)

# validity check
skm.alphabet.check_valid(config["alphabet"])  # raises error if invalid alphabet

output directory:	 learnapp_tutorial_files/learn/output


### Rule 0.5: Unzip files

Any zipped files detected by the above are automatically unzipped. The zipped version of the file is copied into a separate subdirectory.

**Snakemake code:**

    # if any files are gzip compressed, unzip them
    rule unzip:
    input:
        join("input", "{uz}.gz")
    output:
        join("input", "{uz}")
    params:
        outdir=join("input", "zipped")
    shell:
        "mkdir {params.outdir} && gunzip -c {input} > {output} && mv {input} {params.outdir}/."
                

In [6]:
# if any files are gzip compressed, unzip them
for uz in UZS:
    input_ = os.path.join(input_dir, f"{uz}.{UZ_MAP[uz]}.gz")
    output_ = os.path.join(input_dir, f"{uz}.{UZ_MAP[uz]}")
    outdir = os.path.join(input_dir, "zipped")
    
    ! mkdir -p $outdir && gunzip -c $input_ > $output_ && mv $input_ $outdir/.

    print("input:\t", input_)
    print("output:\t", output_)
    

### Rule 1: Preprocess

In this step, we parse user-defined parameters into an appropriate format for subsequent pipeline steps.

Parameter options include:
- `k`: Define kmer length
- `alphabet`: Define the translation alphabet

The Snakemake code is not shown due to length, but the converted Python-ized code is shown below:

In [7]:
for fa in unzipped:
    # this is handled by snakemake but we'll specify it here
    base = f'{skm.utils.split_file_ext(fa)[0]}.kmers'
    output_kmerobj = os.path.join(output_dir, "kmerize", base)
    if not os.path.exists(os.path.join(output_dir, "kmerize")):
        os.mkdir(os.path.join(output_dir, "kmerize"))
        
    base = f'{skm.utils.split_file_ext(fa)[0]}.npz'
    output_data = os.path.join(output_dir, "vector", base)
    if not os.path.exists(os.path.join(output_dir, "vector")):
        os.mkdir(os.path.join(output_dir, "vector"))
    
    fasta = SeqIO.parse(fa, "fasta")

    # initialize kmerization object
    kmer = skm.vectorize.KmerVec(alphabet=config["alphabet"], k=config["k"])

    vecs, seqs, ids, lengths = list(), list(), list(), list()
    for f in fasta:
        vecs.append(kmer.reduce_vectorize(f.seq))
        seqs.append(
            skm.vectorize.reduce(
                f.seq,
                alphabet=config["alphabet"],
                mapping=skm.alphabet.FULL_ALPHABETS,
            )
        )
        ids.append(f.id)
        lengths.append(len(f.seq))

    # save seqIO output and transformed vecs
    np.savez_compressed(output_data, ids=ids, seqs=seqs, vecs=vecs, lengths=lengths)

    with open(output_kmerobj, "wb") as f:
        pickle.dump(kmer, f)

/Users/nitk592/anaconda3/envs/snekmer/lib/python3.10/site-packages/numpy/lib/npyio.py:713: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  val = np.asanyarray(val)


### Rule 2: Learn

In this step, we generate kmer count matrices for each fasta input file. The kmer count matrices contain a kmer count vector for each family, and are merged in the following step to find cumulative kmer counts. These kmer counts can be thought of as training the annotation model.

In [8]:
if not os.path.exists("learnapp_tutorial_files/learn/output/learn"):
    os.makedirs("learnapp_tutorial_files/learn/output/learn")

for fa in unzipped:
    annot_files = glob(os.path.join("learnapp_tutorial_files/learn/annotations", "*.ann"))
    base = f'{skm.utils.split_file_ext(fa)[0]}.npz'
    input_data = os.path.join(output_dir, "vector", base)
    
    Annotation = list()
    for f in annot_files:
        Annotation.append(pd.read_table(f))
    annotations = pd.concat(Annotation)
    Seq_Anot = {}
    Seqs= Annotation[0]['id'].tolist()
    ANNs = Annotation[0]['Family'].tolist()
    for i,seqid in enumerate(Seqs):
        Seq_Anot[seqid] = ANNs[i]
    Seqs = set(Seqs)
    ANNs = set(ANNs)

    output_data = os.path.join(output_dir, "vector", base)
    fasta = SeqIO.parse(fa, "fasta")
    # initialize kmerization object
    kmer = skm.vectorize.KmerVec(alphabet=config["alphabet"], k=config["k"])

    vecs, seqs, ids, lengths = list(), list(), list(), list()

    for f in fasta:
        vecs.append(kmer.reduce_vectorize(f.seq))
        seqs.append(
            skm.vectorize.reduce(
                f.seq,
                alphabet=config["alphabet"],
                mapping=skm.alphabet.FULL_ALPHABETS,
            )
        )
        ids.append(f.id)
        lengths.append(len(f.seq))
        
    df, kmerlist = skm.vectorize.make_feature_matrix(vecs)

    seqids = ids
    kmer_totals = []
    for item in kmerlist:
        kmer_totals.append(0)

    ##### Generate Kmer Counts
    k_len = len(kmerlist[0])
    seq_kmer_dict = {}
    counter = 0
    for i,seq in enumerate(seqids):
        v = seqs[i]
        kmer_counts = dict()
        items = []
        for item in range(0,(len((v)) - k_len +1)):
            items.append(v[item:(item+k_len)])
        for j in items:
            kmer_counts[j] = kmer_counts.get(j, 0) + 1  
        store = []
        for i,item in enumerate(kmerlist):
            if item in kmer_counts:
                store.append(kmer_counts[item])
                kmer_totals[i] += kmer_counts[item]
            else:
                store.append(0)
        seq_kmer_dict[seq]= store


    #Filter out Non-Training Annotations 
    Annotation_Counts = {}
    total_seqs = len(seq_kmer_dict)
    for i,seqid in enumerate(list(seq_kmer_dict)):
        x =re.findall(r'\|(.*?)\|', seqid)[0]
        if x not in Seqs:
            del seq_kmer_dict[seqid]
        else:
            if Seq_Anot[x] not in seq_kmer_dict:
                seq_kmer_dict[Seq_Anot[x]] = seq_kmer_dict.pop(seqid)
            else:
                zipped_lists = zip(seq_kmer_dict.pop(seqid), seq_kmer_dict[Seq_Anot[x]])
                seq_kmer_dict[Seq_Anot[x]] = [x + y for (x, y) in zipped_lists]
            if Seq_Anot[x] not in Annotation_Counts:
                Annotation_Counts[Seq_Anot[x]] = 1
            else: 
                Annotation_Counts[Seq_Anot[x]] += 1

    #Construct Kmer Counts Output
    Kmer_Counts = pd.DataFrame(seq_kmer_dict.values())        
    Kmer_Counts.insert(0,"Annotations",Annotation_Counts.values(),True)
    Kmer_Counts.insert(1,"Kmer Count",(Kmer_Counts[list(Kmer_Counts.columns[1:])].sum(axis=1).to_list()),True)
    kmer_totals[0:0] = [0,total_seqs]
    colnames = ["Sequence count"] + ["Kmer Count"] + list(kmerlist)
    Kmer_Counts = pd.DataFrame(np.insert(Kmer_Counts.values, 0, values=(kmer_totals), axis=0))
    Kmer_Counts.columns = colnames
    new_index = ["Totals"] + list(Annotation_Counts.keys())
    Kmer_Counts.index = new_index
    print("Counts Data Generated for: ",input_data)


    #### Write Output
    out_name = "learnapp_tutorial_files/learn/output/learn/kmer-counts-" + str(input_data)[44:-4] + ".csv"
    Kmer_Counts_out = pa.Table.from_pandas(Kmer_Counts,preserve_index=True)
    csv.write_csv(Kmer_Counts_out, out_name)

Counts Data Generated for:  learnapp_tutorial_files/learn/output/vector/uniprot_reduced.npz


### Rule 3: Merge

In this step, we merge all of the previously generated counts datafiles. While running this pipeline through the command line, we also have the option of merging in a previously generated counts file. This allows for additive kmer count integration for massive project scaling.

In [9]:
input_counts = glob("learnapp_tutorial_files/learn/output/learn/*")

for file_num,f in enumerate(input_counts):
    print("databases merged: ",file_num,"\n")
    Kmer_Counts = pd.read_csv(str(f), index_col="__index_level_0__", header=0, engine="pyarrow")
#     print(Kmer_Counts)
    if file_num == 0:
        running_merge = Kmer_Counts
    elif file_num >= 1:
        running_merge = (pd.concat([running_merge,Kmer_Counts]).reset_index().groupby('__index_level_0__', sort=False).sum(min_count=1)).fillna(0)

running_merge_out = pa.Table.from_pandas(running_merge,preserve_index=True)
csv.write_csv(running_merge_out, "learnapp_tutorial_files/learn/output/learn/kmer-counts-total.csv")

databases merged:  0 



### Rule 4: Eval_Apply

In this step, we find the cosine similarity score between the merged kmer count database and each kmer counts from each sequence in the fasta files. Essentially, we are comparing self-predictions against actual values. This output is used in the next step to calculate confidence scores.

In [10]:
if not os.path.exists("learnapp_tutorial_files/learn/output/eval_apply"):
    os.makedirs("learnapp_tutorial_files/learn/output/eval_apply")

compare_associations = "learnapp_tutorial_files/learn/output/learn/kmer-counts-total.csv"
annotation = ["learnapp_tutorial_files/learn/annotations/TIGRFAMs_annotation.ann"]
for fa in unzipped:
    # this is handled by snakemake but we'll specify it here

    ###
    base = f'{skm.utils.split_file_ext(fa)[0]}.npz'
    output_data = os.path.join(output_dir, "vector", base)

    fasta = SeqIO.parse(fa, "fasta")

    # initialize kmerization object
    kmer = skm.vectorize.KmerVec(alphabet=config["alphabet"], k=config["k"])

    vecs, seqs, ids, lengths = list(), list(), list(), list()

    for f in fasta:
        vecs.append(kmer.reduce_vectorize(f.seq))
        seqs.append(
            skm.vectorize.reduce(
                f.seq,
                alphabet=config["alphabet"],
                mapping=skm.alphabet.FULL_ALPHABETS,
            )
        )
        ids.append(f.id)
        lengths.append(len(f.seq))


    ##### Generate Inputs
    Annotation = list()
    Kmer_Count_Totals = pd.read_csv(str(compare_associations), index_col="__index_level_0__", header=0, engine="c")
    for f in annotation:
        Annotation.append(pd.read_table(f))
    Seqs = Annotation[0]['id'].tolist()
    ANNs = Annotation[0]['Family'].tolist()
    Seq_Anot = {}
    for i,seqid in enumerate(Seqs):
        Seq_Anot[seqid] = ANNs[i]
    Seqs = set(Seqs)
    ANNs = set(ANNs)
#     df, kmerlist = skm.io.load_npz(input.data)
    seqids = ids
    kmer_totals = []
    for item in kmerlist:
        kmer_totals.append(0)

    ##### Generate Kmer Counts
    seq_kmer_dict = {}
    counter = 0
    k_len = len(kmerlist[0])
    for i,seq in enumerate(seqids):
        v = seqs[i]
        kmer_counts = dict()
        items = []
        for item in range(0,(len((v)) - k_len +1)):
            items.append(v[item:(item+k_len)])
        for j in items:
            kmer_counts[j] = kmer_counts.get(j, 0) + 1  
        store = []
        for i,item in enumerate(kmerlist):
            if item in kmer_counts:
                store.append(kmer_counts[item])
                kmer_totals[i] += kmer_counts[item]
            else:
                store.append(0)
        seq_kmer_dict[seq]= store


    ###### ADD Known / Unknown tag to mark for confidence assessment
    Annotation_Counts = {}
    total_seqs = len(seq_kmer_dict)
    count = 0
    for seqid in list(seq_kmer_dict):
        x =re.findall(r'\|(.*?)\|', seqid)[0]
        if x not in Seqs:
            seq_kmer_dict[(x+"_unknown_"+str(count))] = seq_kmer_dict.pop(seqid)
        else: 
            seq_kmer_dict[(Seq_Anot[x] + "_known_" + str(count))] = seq_kmer_dict.pop(seqid)
        count +=1
    Annotation_Counts = {}
    total_seqs = len(seq_kmer_dict)

    ######  Construct Kmer Counts Dataframe
    Kmer_Counts = pd.DataFrame(seq_kmer_dict.values())        
    Kmer_Counts.insert(0,"Annotations",1,True)
    kmer_totals.insert(0,total_seqs)
    Kmer_Counts = pd.DataFrame(np.insert(Kmer_Counts.values, 0, values=kmer_totals, axis=0))
    Kmer_Counts.columns = ["Sequence count"] + list(kmerlist)
    Kmer_Counts.index = ["Totals"] + list(seq_kmer_dict.keys())


    ##### Make New Counts Data match Kmer Counts Totals Format
    if len(str(Kmer_Counts.columns.values[10])) == len(str(Kmer_Count_Totals.columns.values[10])):
        compare_check = True
    else: 
        compare_check = False
    if compare_check == True:
        check_1 = len(Kmer_Counts.columns.values)
        alphabet_initial = set(itertools.chain(*[list(x) for x in Kmer_Counts.columns.values[10:check_1]]))
        alphabet_compare = set(itertools.chain(*[list(x) for x in Kmer_Count_Totals.columns.values[10:check_1]]))
        if alphabet_compare == alphabet_initial:
            compare_check = True
        else: 
            compare_check = False
    if compare_check == False:
        print("Compare Check Failed. ")
        sys.exit()

    new_cols = set(Kmer_Counts.columns)
    compare_cols = set(Kmer_Count_Totals.columns)
    add_to_compare = []
    add_to_new = []
    for val in new_cols:
        if val not in compare_cols:
            add_to_compare.append(val)
    for val in compare_cols:
        if val not in new_cols:
            add_to_new.append(val)

    Kmer_Count_Totals = pd.concat([Kmer_Count_Totals, pd.DataFrame(dict.fromkeys(add_to_compare, 0), index=Kmer_Count_Totals.index)], axis=1)
    Kmer_Count_Totals.drop(columns=Kmer_Count_Totals.columns[:2], index="Totals", axis=0, inplace=True)
    Kmer_Counts = pd.concat([Kmer_Counts, pd.DataFrame(dict.fromkeys(add_to_new, 0), index=Kmer_Counts.index)], axis=1)
    Kmer_Counts.drop(columns=Kmer_Counts.columns[-1:].union(Kmer_Counts.columns[:1]), index="Totals", axis=0, inplace=True)

    #Perform Cosine Similarity between Kmer Counts Totals and Counts and Sums DF
    cosine_df = sklearn.metrics.pairwise.cosine_similarity(Kmer_Count_Totals,Kmer_Counts).T
    final_matrix_with_scores = pd.DataFrame(cosine_df, columns=Kmer_Count_Totals.index, index=Kmer_Counts.index)

    #Write Output
    out_name = "learnapp_tutorial_files/learn/output/eval_apply/seq-annotation-scores-" + str(fa)[36:-6] + ".csv"

    final_matrix_with_scores_write = pa.Table.from_pandas(final_matrix_with_scores)
    csv.write_csv(final_matrix_with_scores_write, out_name)
    print("File completed: seq-annotation-scores-" + str(fa)[36:-6] + ".csv")



File completed: seq-annotation-scores-uniprot_reduced.csv


### Rule 5: Eval_Apply_Reverse_Sequences

We repeat the previous step using reversed sequences, allowing us to calculate family thresholds from the distribution of the resulting cosine similarity scores.

In [11]:
if not os.path.exists("learnapp_tutorial_files/learn/output/eval_apply_reversed"):
    os.makedirs("learnapp_tutorial_files/learn/output/eval_apply_reversed")

compare_associations = "learnapp_tutorial_files/learn/output/learn/kmer-counts-total.csv"
annotation = ["learnapp_tutorial_files/learn/annotations/TIGRFAMs_annotation.ann"]
for fa in unzipped:
    # this is handled by snakemake but we'll specify it here

    ###
    base = f'{skm.utils.split_file_ext(fa)[0]}.npz'
    output_data = os.path.join(output_dir, "vector", base)

    fasta = SeqIO.parse(fa, "fasta")

    # initialize kmerization object
    kmer = skm.vectorize.KmerVec(alphabet=config["alphabet"], k=config["k"])

    vecs, seqs, ids, lengths = list(), list(), list(), list()

    for f in fasta:
        vecs.append(kmer.reduce_vectorize(f.seq))
        seqs.append(
            skm.vectorize.reduce(
                f.seq,
                alphabet=config["alphabet"],
                mapping=skm.alphabet.FULL_ALPHABETS,
            )
        )
        ids.append(f.id)
        lengths.append(len(f.seq))


    ##### Generate Inputs
    Annotation = list()
    Kmer_Count_Totals = pd.read_csv(str(compare_associations), index_col="__index_level_0__", header=0, engine="c")
    for f in annotation:
        Annotation.append(pd.read_table(f))
    Seqs = Annotation[0]['id'].tolist()
    ANNs = Annotation[0]['Family'].tolist()
    Seq_Anot = {}
    for i,seqid in enumerate(Seqs):
        Seq_Anot[seqid] = ANNs[i]
    Seqs = set(Seqs)
    ANNs = set(ANNs)
#     df, kmerlist = skm.io.load_npz(input.data)
    seqids = ids
    kmer_totals = []
    for item in kmerlist:
        kmer_totals.append(0)

    ##### Generate Kmer Counts
    seq_kmer_dict = {}
    counter = 0
    k_len = len(kmerlist[0])
    for i,seq in enumerate(seqids):
        v = seqs[i]
        v = v[::-1]
        kmer_counts = dict()
        items = [v[item : (item + k_len)]
                for item in range(0, (len((v)) - k_len + 1))]
        for item in range(0,(len((v)) - k_len +1)):
            items.append(v[item:(item+k_len)])
        for j in items:
            kmer_counts[j] = kmer_counts.get(j, 0) + 1  
        store = []
        for i,item in enumerate(kmerlist):
            if item in kmer_counts:
                store.append(kmer_counts[item])
                kmer_totals[i] += kmer_counts[item]
            else:
                store.append(0)
        seq_kmer_dict[seq]= store


    ###### ADD Known / Unknown tag to mark for confidence assessment
    Annotation_Counts = {}
    total_seqs = len(seq_kmer_dict)
    count = 0
    for seqid in list(seq_kmer_dict):
        x =re.findall(r'\|(.*?)\|', seqid)[0]
        if x not in Seqs:
            seq_kmer_dict[(x+"_unknown_"+str(count))] = seq_kmer_dict.pop(seqid)
        else: 
            seq_kmer_dict[(Seq_Anot[x] + "_known_" + str(count))] = seq_kmer_dict.pop(seqid)
        count +=1
    Annotation_Counts = {}
    total_seqs = len(seq_kmer_dict)

    ######  Construct Kmer Counts Dataframe
    Kmer_Counts = pd.DataFrame(seq_kmer_dict.values())        
    Kmer_Counts.insert(0,"Annotations",1,True)
    kmer_totals.insert(0,total_seqs)
    Kmer_Counts = pd.DataFrame(np.insert(Kmer_Counts.values, 0, values=kmer_totals, axis=0))
    Kmer_Counts.columns = ["Sequence count"] + list(kmerlist)
    Kmer_Counts.index = ["Totals"] + list(seq_kmer_dict.keys())


    ##### Make New Counts Data match Kmer Counts Totals Format
    if len(str(Kmer_Counts.columns.values[10])) == len(str(Kmer_Count_Totals.columns.values[10])):
        compare_check = True
    else: 
        compare_check = False
    if compare_check == True:
        check_1 = len(Kmer_Counts.columns.values)
        alphabet_initial = set(itertools.chain(*[list(x) for x in Kmer_Counts.columns.values[10:check_1]]))
        alphabet_compare = set(itertools.chain(*[list(x) for x in Kmer_Count_Totals.columns.values[10:check_1]]))
        if alphabet_compare == alphabet_initial:
            compare_check = True
        else: 
            compare_check = False
    if compare_check == False:
        print("Compare Check Failed. ")
        sys.exit()

    new_cols = set(Kmer_Counts.columns)
    compare_cols = set(Kmer_Count_Totals.columns)
    add_to_compare = []
    add_to_new = []
    for val in new_cols:
        if val not in compare_cols:
            add_to_compare.append(val)
    for val in compare_cols:
        if val not in new_cols:
            add_to_new.append(val)

    Kmer_Count_Totals = pd.concat([Kmer_Count_Totals, pd.DataFrame(dict.fromkeys(add_to_compare, 0), index=Kmer_Count_Totals.index)], axis=1)
    Kmer_Count_Totals.drop(columns=Kmer_Count_Totals.columns[:2], index="Totals", axis=0, inplace=True)
    Kmer_Counts = pd.concat([Kmer_Counts, pd.DataFrame(dict.fromkeys(add_to_new, 0), index=Kmer_Counts.index)], axis=1)
    Kmer_Counts.drop(columns=Kmer_Counts.columns[-1:].union(Kmer_Counts.columns[:1]), index="Totals", axis=0, inplace=True)

    #Perform Cosine Similarity between Kmer Counts Totals and Counts and Sums DF
    cosine_df = sklearn.metrics.pairwise.cosine_similarity(Kmer_Count_Totals,Kmer_Counts).T
    final_matrix_with_scores = pd.DataFrame(cosine_df, columns=Kmer_Count_Totals.index, index=Kmer_Counts.index)

    #Write Output
    out_name = "learnapp_tutorial_files/learn/output/eval_apply_reversed/seq-annotation-scores-" + str(fa)[36:-6] + ".csv"

    final_matrix_with_scores_write = pa.Table.from_pandas(final_matrix_with_scores)
    csv.write_csv(final_matrix_with_scores_write, out_name)
    print("File completed: seq-annotation-scores-" + str(fa)[36:-6] + ".csv")



File completed: seq-annotation-scores-uniprot_reduced.csv


### Rule 6: Eval_Conf

In this step, we evaluate if the cosine scores between kmer counts dataframes accurately predict the correct annotation. The ratio of true positive to false positives is taken and we generate our global confidence scores.  Each delta will be mapped to a confidence score. **Delta** is defined as the difference between the two highest cosine similarity scores. 

In [12]:
if not os.path.exists("learnapp_tutorial_files/learn/output/eval_conf"):
    os.makedirs("learnapp_tutorial_files/learn/output/eval_conf")


eval_apply_data = glob("learnapp_tutorial_files/learn/output/eval_apply/seq-annotation-scores-*")
        #### Generate Input Data
for j,f in enumerate(eval_apply_data):
    seq_ann_scores = pd.read_csv(f, index_col="__index_level_0__", header=0, engine="c")
    max_value_index = seq_ann_scores.idxmax(axis="columns")
    result = max_value_index.keys()
    TF = list()
    Known = list()
    for i,item in enumerate(list(max_value_index)):
        if item in result[i]:
            TF.append("T")
        else:
            TF.append("F")
        if "unknown" in result[i]:
            Known.append("Unknown")
        else:
            Known.append("Known")

    seq_ann_vals = seq_ann_scores.values
    seq_ann_vals = seq_ann_scores.values[np.arange(len(seq_ann_scores))[:,None],np.argpartition(-seq_ann_vals,np.arange(2),axis=1)[:,:2]]

    diff_df = pd.DataFrame(seq_ann_vals, columns = ['Top','Second'])
    diff_df['Delta'] = -(np.diff(seq_ann_vals, axis=1).round(decimals=2))
    diff_df['Prediction'] = list(max_value_index)
    diff_df['Actual'] = result
    diff_df["T/F"] = TF
    diff_df["Known/Unknown"] = Known

    #### Create CrossTabs - ie True/False Count Sums and sum within .01 intervals
    known_true_diff_df = diff_df[(diff_df["Known/Unknown"] == "Known") & (diff_df["T/F"] == "T")]
    known_false_diff_df = diff_df[(diff_df["Known/Unknown"] == "Known") & (diff_df["T/F"] == "F")]
    possible_vals = [round(x * 0.01,2) for x in range(0, 101)]
    true_crosstab = pd.crosstab(known_true_diff_df.Prediction,known_true_diff_df.Delta)
    false_crosstab = pd.crosstab(known_false_diff_df.Prediction,known_false_diff_df.Delta)

    if j == 0:
        true_running_crosstab = true_crosstab
        false_running_crosstab = false_crosstab
    else:
        true_running_crosstab = (pd.concat([true_running_crosstab,true_crosstab]).reset_index().groupby('Prediction', sort=False).sum(min_count=1)).fillna(0)
        false_running_crosstab = (pd.concat([false_running_crosstab,false_crosstab]).reset_index().groupby('Prediction', sort=False).sum(min_count=1)).fillna(0)


    add_to_true_df = pd.DataFrame(0, index = sorted(set(false_running_crosstab.index) - set(true_running_crosstab.index)), columns= true_running_crosstab.columns)
    add_to_false_df = pd.DataFrame(0, index = sorted(set(true_running_crosstab.index) - set(false_running_crosstab.index)), columns= false_running_crosstab.columns)

    true_running_crosstab = pd.concat([true_running_crosstab,add_to_true_df])[sorted(list(set(possible_vals) & set(true_running_crosstab.columns)))].assign(**dict.fromkeys(list(map(str, sorted(list(set(possible_vals) ^ set(true_running_crosstab.columns.astype(float)))))),0))
    false_running_crosstab = pd.concat([false_running_crosstab,add_to_false_df])[sorted(list(set(possible_vals) & set(false_running_crosstab.columns)))].assign(**dict.fromkeys(list(map(str, sorted(list(set(possible_vals) ^ set(false_running_crosstab.columns.astype(float)))))),0))

    true_running_crosstab.index.names = ['Prediction']
    false_running_crosstab.index.names = ['Prediction']
    true_running_crosstab.sort_index(inplace=True) 
    false_running_crosstab.sort_index(inplace=True) 
    true_running_crosstab.columns = true_running_crosstab.columns.astype(float)
    false_running_crosstab.columns = false_running_crosstab.columns.astype(float)
    true_running_crosstab = true_running_crosstab[sorted(true_running_crosstab.columns)]
    false_running_crosstab = false_running_crosstab[sorted(false_running_crosstab.columns)]


    print("Dataframes joined: ", j+1, " out of ",len(eval_apply_data) , ".")

#### Generate Each Global CrossTab
ratio_running_crosstab = (true_running_crosstab/(true_running_crosstab + false_running_crosstab))
true_total_dist = true_running_crosstab.sum(numeric_only=True, axis=0)
false_total_dist = false_running_crosstab.sum(numeric_only=True, axis=0)
ratio_total_dist = (true_running_crosstab.sum(numeric_only=True, axis=0)/(true_running_crosstab.sum(numeric_only=True, axis=0) + false_running_crosstab.sum(numeric_only=True, axis=0)))

####Interpolate For final Ratio, this only will affect upper limit values if there is a decent amount of data
ratio_total_dist = ratio_total_dist.interpolate(method="linear")

##### Write Final Confidence Results
ratio_total_dist.to_csv("learnapp_tutorial_files/learn/output/eval_conf/global-confidence-scores.csv")
csv.write_csv(pa.Table.from_pandas(true_running_crosstab), "learnapp_tutorial_files/learn/output/eval_conf/true-total.csv")
csv.write_csv(pa.Table.from_pandas(false_running_crosstab), "learnapp_tutorial_files/learn/output/eval_conf/false-total.csv")
csv.write_csv(pa.Table.from_pandas(ratio_running_crosstab), "learnapp_tutorial_files/learn/output/eval_conf/confidence-matrix.csv")

print("\nGlobal Confidence scores mapped to Delta:\n", ratio_total_dist)


Dataframes joined:  1  out of  1 .

Global Confidence scores mapped to Delta:
 Delta
0.00    0.666667
0.01    1.000000
0.02    1.000000
0.03    1.000000
0.04    1.000000
          ...   
0.96    1.000000
0.97    1.000000
0.98    1.000000
0.99    1.000000
1.00    1.000000
Length: 101, dtype: float64


/var/folders/x_/kpwh_b592s74xnvx5r318c6c0000gn/T/ipykernel_70133/2779374347.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  false_running_crosstab = pd.concat([false_running_crosstab,add_to_false_df])[sorted(list(set(possible_vals) & set(false_running_crosstab.columns)))].assign(**dict.fromkeys(list(map(str, sorted(list(set(possible_vals) ^ set(false_running_crosstab.columns.astype(float)))))),0))


### Rule 7: ReverseDecoy_Evaluations

In the last step of the Learn pipeline, we take the scores obtained by applying the kmer count associations to reversed sequences and perform statistical calculations to produce each of the family score thresholds that may be selected by the user. When running Learn from the command line, this step also allows the results to be combined with those from an earlier run.

In [13]:
if not os.path.exists("learnapp_tutorial_files/learn/output/eval_apply_reversed"):
    os.makedirs("learnapp_tutorial_files/learn/output/eval_apply_reversed")

eval_apply_reversed_data = glob("learnapp_tutorial_files/learn/output/eval_apply_reversed/seq-annotation-scores-*")

chunk_size = 10000  # Adjust based on available memory
reservoir_size = 100000  # Size of the reservoir for percentiles
for file in eval_apply_reversed_data:
    stats = {}
    for chunk in pd.read_csv(file, chunksize=chunk_size):
        families = chunk.columns[:-1]
        for family in families:
            values = chunk[family].dropna().astype(float).values
            n = len(values)
            stats[family] = {
                    "count": 0,
                    "sum": 0.0,
                    "sumSqr": 0.0,
                    "min": np.inf,
                    "max": -np.inf,
                    "values_for_percentiles": [],
                }
            stats[family]["sum"] += values.sum()
            stats[family]["sumSqr"] += np.dot(values, values)
            stats[family]["min"] = min(stats[family]["min"], values.min())
            stats[family]["max"] = max(stats[family]["max"], values.max())
            # Reservoir sampling for percentiles
            for value in values:
                stats[family]["count"] += 1  # Update total count
                total_seen = stats[family]["count"]
                if len(stats[family]["values_for_percentiles"]) < reservoir_size:
                    # Fill the reservoir until it reaches the desired size
                    stats[family]["values_for_percentiles"].append(value)
                else:
                    # Replace elements with decreasing probability
                    j = random.randint(0, total_seen - 1)
                    if j < reservoir_size:
                        stats[family]["values_for_percentiles"][j] = value
statsData = {
    "Family": [],
    "Mean": [],
    "Std Dev": [],
    "Min": [],
    "10th Percentile": [],
    "20th Percentile": [],
    "25th Percentile": [],
    "30th Percentile": [],
    "40th Percentile": [],
    "Median": [],
    "60th Percentile": [],
    "70th Percentile": [],
    "75th Percentile": [],
    "80th Percentile": [],
    "90th Percentile": [],
    "Max": [],
    "1 Std Dev Above": [],
    "1 Std Dev Below": [],
    "2 Std Dev Above": [],
    "2 Std Dev Below": [],
    }
for family, fam_stats in stats.items():
    n = fam_stats["count"]
    sum_ = fam_stats["sum"]
    sumSqr = fam_stats["sumSqr"]
    mean = sum_ / n if n > 0 else 0.0
    variance = (sumSqr - (sum_**2) / n) / (n - 1) if n > 1 else 0.0
    std_dev = np.sqrt(variance)
    values = np.array(fam_stats["values_for_percentiles"])
    if len(values) > 0:
        percentiles = np.percentile(
            values, [10, 20, 25, 30, 40, 50, 60, 70, 75, 80, 90]
        )
    else:
        # If no values, fill with NaN
        percentiles = [np.nan] * 11
    statsData["Family"].append(family)
    statsData["Mean"].append(round(mean, 3))
    statsData["Std Dev"].append(round(std_dev, 3))
    statsData["Min"].append(round(fam_stats["min"], 3))
    statsData["10th Percentile"].append(round(percentiles[0], 3))
    statsData["20th Percentile"].append(round(percentiles[1], 3))
    statsData["25th Percentile"].append(round(percentiles[2], 3))
    statsData["30th Percentile"].append(round(percentiles[3], 3))
    statsData["40th Percentile"].append(round(percentiles[4], 3))
    statsData["Median"].append(round(percentiles[5], 3))
    statsData["60th Percentile"].append(round(percentiles[6], 3))
    statsData["70th Percentile"].append(round(percentiles[7], 3))
    statsData["75th Percentile"].append(round(percentiles[8], 3))
    statsData["80th Percentile"].append(round(percentiles[9], 3))
    statsData["90th Percentile"].append(round(percentiles[10], 3))
    statsData["Max"].append(round(fam_stats["max"], 3))
    statsData["1 Std Dev Above"].append(round(mean + std_dev, 3))
    statsData["1 Std Dev Below"].append(round(mean - std_dev, 3))
    statsData["2 Std Dev Above"].append(round(mean + 2 * std_dev, 3))
    statsData["2 Std Dev Below"].append(round(mean - 2 * std_dev, 3))

familyStatisticsDf = pd.DataFrame(statsData)
familyStatisticsDf.to_csv(os.path.join("learnapp_tutorial_files/learn/output/eval_conf", "family_summary_stats.csv"), index=False)

### Learn Pipeline is done.

Key outputs include:  
* Kmer counts database: /output/learn/kmer-counts-total.csv  
* Global confidence scores: output/eval_conf/global-confidence-scores.csv
* Statistics of scores for reversed sequences against each family: /output/eval_conf/family_summary_stats.csv

The next step is to prepare for the Apply pipeline.

### Intermediate Steps

Users will have to extract key outputs and copy them into a new directory to run the Apply pipeline. When running from the command line all of the needed inputs will be automatically copied to ```output/apply_inputs```, but we're only going to copy them once here.


In [14]:
if not os.path.exists("learnapp_tutorial_files/apply/counts"):
    os.makedirs("learnapp_tutorial_files/apply/counts")

if not os.path.exists("learnapp_tutorial_files/apply/confidence"):
    os.makedirs("learnapp_tutorial_files/apply/confidence")

if not os.path.exists("learnapp_tutorial_files/apply/stats"):
    os.makedirs("learnapp_tutorial_files/apply/stats")
    
shutil.copyfile("learnapp_tutorial_files/learn/output/learn/kmer-counts-total.csv", "learnapp_tutorial_files/apply/counts/kmer-counts-total.csv")
shutil.copyfile("learnapp_tutorial_files/learn/output/eval_conf/global-confidence-scores.csv", "learnapp_tutorial_files/apply/confidence/global-confidence-scores.csv")
shutil.copyfile("learnapp_tutorial_files/learn/output/eval_conf/family_summary_stats.csv", "learnapp_tutorial_files/apply/stats/family_summary_stats.csv")

'learnapp_tutorial_files/apply/stats/family_summary_stats.csv'

## Getting Started with Snekmer Apply

### Setup


Before running Snekmer Apply, verify that files have been placed in an **_input_** directory placed at the same level as the **_config.yaml_** file. The assumed file directory structure is illustrated below.

    .
    ├── input
    │   ├── W.fasta
    │   ├── X.fasta
    │   ├── Y.fasta
    │   ├── Z.fasta
    │   └── etc.
    ├── config.yaml
    ├── counts
    │   └── kmer-counts-total.csv
    ├── confidence
    │    └── global-confidence-scores.csv
    └── stats
        └── family_summary_stats.csv
     
        
    
Note: Snekmer automatically creates the **_output_** directory when creating output files, so there is no need to create this folder in advance



## Running Snekmer Apply Pipeline

### Rule 0.5: Unzip files

Any zipped files detected by the above are automatically unzipped. The zipped version of the file is copied into a separate subdirectory.
                

In [15]:
# if any files are gzip compressed, unzip them
for uz in UZS:
    input_ = os.path.join(input_dir, f"{uz}.{UZ_MAP[uz]}.gz")
    output_ = os.path.join(input_dir, f"{uz}.{UZ_MAP[uz]}")
    outdir = os.path.join(input_dir, "zipped")
    
    ! mkdir -p $outdir && gunzip -c $input_ > $output_ && mv $input_ $outdir/.

    print("input:\t", input_)
    print("output:\t", output_)
    

### kmerize/vectorize
for fa in unzipped:
    # this is handled by snakemake but we'll specify it here
    base = f'{skm.utils.split_file_ext(fa)[0]}.kmers'
    output_kmerobj = os.path.join(output_dir, "kmerize", base)
    if not os.path.exists(os.path.join(output_dir, "kmerize")):
        os.mkdir(os.path.join(output_dir, "kmerize"))
        
    base = f'{skm.utils.split_file_ext(fa)[0]}.npz'
    output_data = os.path.join(output_dir, "vector", base)
    if not os.path.exists(os.path.join(output_dir, "vector")):
        os.mkdir(os.path.join(output_dir, "vector"))
    
    fasta = SeqIO.parse(fa, "fasta")

    # initialize kmerization object
    kmer = skm.vectorize.KmerVec(alphabet=config["alphabet"], k=config["k"])

    vecs, seqs, ids, lengths = list(), list(), list(), list()
    for f in fasta:
        vecs.append(kmer.reduce_vectorize(f.seq))
        seqs.append(
            skm.vectorize.reduce(
                f.seq,
                alphabet=config["alphabet"],
                mapping=skm.alphabet.FULL_ALPHABETS,
            )
        )
        ids.append(f.id)
        lengths.append(len(f.seq))

    # save seqIO output and transformed vecs
    np.savez_compressed(output_data, ids=ids, seqs=seqs, vecs=vecs, lengths=lengths)

    with open(output_kmerobj, "wb") as f:
        pickle.dump(kmer, f)


/Users/nitk592/anaconda3/envs/snekmer/lib/python3.10/site-packages/numpy/lib/npyio.py:713: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  val = np.asanyarray(val)


### Rule 1: Preprocess

In this step, we parse user-defined parameters into an appropriate format for subsequent pipeline steps.

Parameter options include:
- `k`: Define kmer length
- `alphabet`: Define the translation alphabet

Note: This is essentially the same step as in Learn.

In [16]:
# collect all fasta-like files, unzipped filenames, and basenames
input_dir = "learnapp_tutorial_files/apply/input/"
input_files = glob(os.path.join(input_dir, "*"))
zipped = [fa for fa in input_files if fa.endswith(".gz")]
unzipped = [
    fa.rstrip(".gz")
    for fa, ext in itertools.product(input_files, config["input"]["file_extensions"])
    if fa.rstrip(".gz").endswith(f".{ext}")
]

print("zipped files:\t", zipped)
print("unzipped files:\t", unzipped)
# define output directory (and create if missing)
output_dir = "learnapp_tutorial_files/apply/output"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print("output directory:\t", output_dir)

# validity check
skm.alphabet.check_valid(config["alphabet"])  # raises error if invalid alphabet
# if any files are gzip compressed, unzip them
for uz in UZS:
    input_ = os.path.join(input_dir, f"{uz}.{UZ_MAP[uz]}.gz")
    output_ = os.path.join(input_dir, f"{uz}.{UZ_MAP[uz]}")
    outdir = os.path.join(input_dir, "zipped")
    
    ! mkdir -p $outdir && gunzip -c $input_ > $output_ && mv $input_ $outdir/.

    print("input:\t", input_)
    print("output:\t", output_)
    

zipped files:	 []
unzipped files:	 ['learnapp_tutorial_files/apply/input/UP000481030_1602942.fasta', 'learnapp_tutorial_files/apply/input/UP000509303_2763006.fasta']
output directory:	 learnapp_tutorial_files/apply/output


### Rule 2: Apply
In this step, we find the cosine similarity score between each family kmer count vector stored in kmer-counts-total.csv and the kmer count vector from each sequence in the FASTA input files. Essentially, we are comparing new kmer count frequencies against the trained model/dataframe. 

 
This compares each new sequence against every sequence have in the trained model.   

Optional output: 
* Specify learnapp parameter 'save_results' = True in the config file.
* This will output a matrix of all cosine similarity scores. 
* **Warning**: these files may take up a lot of storage space.

In [17]:
if not os.path.exists("learnapp_tutorial_files/apply/output/apply"):
    os.makedirs("learnapp_tutorial_files/apply/output/apply")

confidence_associations = "learnapp_tutorial_files/apply/confidence/global-confidence-scores.csv"
compare_associations = "learnapp_tutorial_files/apply/counts/kmer-counts-total.csv"
decoy_stats = "learnapp_tutorial_files/apply/stats/family_summary_stats.csv"



for fa in unzipped:
    # this is handled by snakemake but we'll specify it here

    ###
    base = f'{skm.utils.split_file_ext(fa)[0]}.npz'
    output_data = os.path.join(output_dir, "vector", base)

    print("parsing...")
    fasta = SeqIO.parse(fa, "fasta")
    print("finished...")

    # initialize kmerization object
    kmer = skm.vectorize.KmerVec(alphabet=config["alphabet"], k=config["k"])

    vecs, seqs, ids, lengths = list(), list(), list(), list()

    for f in fasta:
        vecs.append(kmer.reduce_vectorize(f.seq))
        seqs.append(
            skm.vectorize.reduce(
                f.seq,
                alphabet=config["alphabet"],
                mapping=skm.alphabet.FULL_ALPHABETS,
            )
        )
        ids.append(f.id)
        lengths.append(len(f.seq))

    print("making kmer list")
    df, kmerlist = skm.vectorize.make_feature_matrix(vecs)
    print("done")

    kmer_count_totals = pd.read_csv(str(compare_associations), index_col="__index_level_0__", header=0, engine="c")
    seqids = ids
    kmer_totals = []
    for item in kmerlist:
        kmer_totals.append(0)
    k_len = len(kmerlist[0])
        
    
    ##### Generate Kmer Counts
    seq_kmer_dict = {}
    counter = 0
    for i,seq in enumerate(seqids):
        v = seqs[i]
        kmer_counts = dict()
        items = []
        for item in range(0,(len((v)) - k_len +1)):
            items.append(v[item:(item+k_len)])
        for j in items:
            kmer_counts[j] = kmer_counts.get(j, 0) + 1  
        store = []
        for i,item in enumerate(kmerlist):
            if item in kmer_counts:
                store.append(kmer_counts[item])
                kmer_totals[i] += kmer_counts[item]
            else:
                store.append(0)
        seq_kmer_dict[seq]= store


    ######  Construct Kmer Counts Dataframe
    total_seqs = len(seq_kmer_dict)
    kmer_counts = pd.DataFrame(seq_kmer_dict.values())        
    kmer_counts.insert(0,"Annotations",1,True)
    kmer_totals.insert(0,total_seqs)
    kmer_counts = pd.DataFrame(np.insert(kmer_counts.values, 0, values=kmer_totals, axis=0))
    kmer_counts.columns = ["Sequence count"] + list(kmerlist)
    kmer_counts.index = ["Totals"] + list(seq_kmer_dict.keys())

    new_associations = kmer_counts.iloc[1:, 1:].div(kmer_counts["Sequence count"].tolist()[1:], axis = "rows")

    ##### Make Kmer Counts Dataframe match Kmer Counts Totals Format
    if len(str(kmer_counts.columns.values[10])) == len(str(kmer_count_totals.columns.values[10])):
        compare_check = True
    else: 
        compare_check = False
    if compare_check:
        check_1 = len(new_associations.columns.values)
        check_2 = len(kmer_count_totals.columns.values)
        alphabet_initial = set(itertools.chain(*[list(x) for x in kmer_counts.columns.values[10:check_1]]))
        alphabet_compare = set(itertools.chain(*[list(x) for x in kmer_count_totals.columns.values[10:check_1]]))
        if alphabet_compare == alphabet_initial:
            compare_check = True
    else:
        print("Compare Check Failed. ")
        sys.exit()

    new_cols = set(kmer_counts.columns)
    compare_cols = set(kmer_count_totals.columns)
    add_to_compare = []
    add_to_new = []
    for val in new_cols:
        if val not in compare_cols:
            add_to_compare.append(val)
    for val in compare_cols:
        if val not in new_cols:
            add_to_new.append(val)

    kmer_count_totals = pd.concat([kmer_count_totals, pd.DataFrame(dict.fromkeys(add_to_compare, 0), index=kmer_count_totals.index)], axis=1)
    kmer_count_totals.drop(columns=kmer_count_totals.columns[:2], index="Totals", axis=0, inplace=True)
    kmer_counts = pd.concat([kmer_counts, pd.DataFrame(dict.fromkeys(add_to_new, 0), index=kmer_counts.index)], axis=1)
    kmer_counts.drop(columns=kmer_counts.columns[-1:].union(kmer_counts.columns[:1]), index="Totals", axis=0, inplace=True)


    #### Perform Cosine Similarity between Kmer Counts Totals and Counts and Sums DF
    cosine_df = sklearn.metrics.pairwise.cosine_similarity(kmer_count_totals,kmer_counts).T
    kmer_count_totals = pd.DataFrame(cosine_df, columns=kmer_count_totals.index, index=kmer_counts.index)

    #### Select best hit using specified decoy threshold and selection criterion
    if config["learnapp"]["selection"] == "top_hit":
        if config["learnapp"]["threshold_type"] is None:
            for row_id, row in kmer_count_totals.iterrows():
                if not row.empty:
                    sorted_row = row.sort_values(ascending=False)
                    top_value = sorted_row.iloc[0]
                    top_family = sorted_row.index[0]
                if len(sorted_row) > 1:
                    second_value = sorted_row.iloc[1]
                    delta = top_value - second_value
                else:
                    delta = top_value
                self.selected_values[row_id] = (top_family, top_value, delta)
            else:
                self.selected_values[row_id] = (None, None, None)
        else:
            decoy_df = pd.read_csv(str(decoy_stats), header=0, engine="c")
            threshold_type = config["learnapp"]["threshold_type"]
            threshold_dict = dict(zip(decoy_df.Family, decoy_df[threshold_type]))
            selected_values = {}
            filtered_out_count = 0

        for row_id, row in kmer_count_totals.iterrows():
            threshold_values = row.index.map(threshold_dict.get)
            threshold_series = pd.Series(threshold_values, index=row.index)

            row_values = row[row > threshold_series]

            if not row_values.empty:
                sorted_row = row_values.sort_values(ascending=False)
                top_value = sorted_row.iloc[0]
                top_family = sorted_row.index[0]
                if len(sorted_row) > 1:
                    second_value = sorted_row.iloc[1]
                    delta = top_value - second_value
                else:
                    delta = top_value - threshold_dict.get(top_family, 0)
                    selected_values[row_id] = (top_family, top_value, delta)
            else:
                selected_values[row_id] = (None, None, None)
                filtered_out_count += 1
    elif config["learnapp"]["selection"] == "greatest_distance":
        decoy_df = pd.read_csv(str(decoy_stats), header=0, engine="c")
        threshold_type = config["learnapp"]["threshold_type"]
        threshold_dict = dict(zip(decoy_df.Family, decoy_df[threshold_type]))
        selected_values = {}
        filtered_out_count = 0
        for row_id, row in kmer_count_totals.iterrows():
            distances = row - row.index.map(threshold_dict.get)
            positive_distances = distances[distances > 0]

            if not positive_distances.empty:
                greatest_distance_family = positive_distances.idxmax()
                greatest_distance_value = row[greatest_distance_family]
                delta = positive_distances.max()
                selected_values[row_id] = (
                    greatest_distance_family,
                    greatest_distance_value,
                    delta,
                )
            else:
                selected_values[row_id] = (None, None, None)
                filtered_out_count += 1
    elif config["learnapp"]["selection"] == "combined_distance":
        weight_top = config["learnapp"].get("weight_top", 0.5)
        weight_distance = config["learnapp"].get("weight_distance", 0.5)
        decoy_df = pd.read_csv(str(decoy_stats), header=0)
        threshold_type = config["learnapp"]["threshold_type"]
        threshold_dict = dict(zip(decoy_df.Family, decoy_df[threshold_type]))
        selected_values = {}
        filtered_out_count = 0
        for row_id, row in kmer_count_totals.iterrows():
            threshold_values = row.index.map(threshold_dict.get)
            threshold_series = pd.Series(threshold_values, index=row.index)

            row_values_above_threshold = row[row > threshold_series]
            distances = row - threshold_series
            positive_distances = distances[distances > 0]

            candidates = {}

            if not row_values_above_threshold.empty:
                top_value = row_values_above_threshold.max()
                top_family = row_values_above_threshold.idxmax()
                top_threshold = threshold_dict.get(top_family, 0)
                candidates[top_family] = {
                    "value": top_value,
                    "delta": top_value - top_threshold,
                    "score": (top_value * weight_top)
                    + ((top_value - top_threshold) * weight_distance),
                }

            if not positive_distances.empty:
                greatest_distance_family = positive_distances.idxmax()
                greatest_distance_value = row[greatest_distance_family]
                greatest_distance_threshold = threshold_dict.get(
                    greatest_distance_family, 0
                )
                candidates[greatest_distance_family] = {
                    "value": greatest_distance_value,
                    "delta": positive_distances.max(),
                    "score": (greatest_distance_value * weight_top)
                    + (positive_distances.max() * weight_distance),
                }

            if candidates:
                best_candidate = max(
                    candidates.items(), key=lambda x: x[1]["score"]
                )
                selected_values[row_id] = (
                    best_candidate[0],
                    best_candidate[1]["value"],
                    best_candidate[1]["delta"],
                )
            else:
                selected_values[row_id] = (None, None, None)
                filtered_out_count += 1
    


    ##### Format and write output
    output_kmer_summary = "learnapp_tutorial_files/apply/output/apply/kmer-summary-" + str(fa)[36:-6] + ".csv"
    if config["learnapp"]["save_apply_associations"]:
        kmer_count_totals_write = pa.Table.from_pandas(
            kmer_count_totals
        )
        csv.write_csv(kmer_count_totals_write, output_seq_ann)

    global_confidence_scores = pd.read_csv(
        str(confidence_associations)
    )
    global_confidence_scores.index = global_confidence_scores[
        global_confidence_scores.columns[0]
    ]
    global_confidence_scores = global_confidence_scores.iloc[:, 1:]
    global_confidence_scores = global_confidence_scores[
        global_confidence_scores.columns[0]
    ].squeeze()

    results_list = []
    for row_id in kmer_count_totals.index:
        if row_id in selected_values:
            prediction, score, delta = selected_values[row_id]
            if delta is None:
                delta = 0
        else:
            prediction, score, delta = None, None, 0
        results_list.append(
            {
                "Sequence": row_id,
                "Prediction": prediction,
                "Score": score,
                "delta": round(delta, 2),
            }
        )

    results = pd.DataFrame(results_list)
    results.set_index("Sequence", inplace=True)

    results["Confidence"] = results["delta"].map(global_confidence_scores)

    results.reset_index(inplace=True)
    results_write = pa.Table.from_pandas(results)
    csv.write_csv(results_write, output_kmer_summary)
    
    print(results)


parsing...
finished...
making kmer list
done
                            Sequence Prediction     Score  delta  Confidence
0     tr|A0A6L3UVZ1|A0A6L3UVZ1_9BACI       None       NaN   0.00    0.666667
1     tr|A0A6L3UWN7|A0A6L3UWN7_9BACI       None       NaN   0.00    0.666667
2     tr|A0A6L3UX61|A0A6L3UX61_9BACI       None       NaN   0.00    0.666667
3     tr|A0A6L3UXS8|A0A6L3UXS8_9BACI       None       NaN   0.00    0.666667
4     tr|A0A6L3UXV5|A0A6L3UXV5_9BACI       None       NaN   0.00    0.666667
...                              ...        ...       ...    ...         ...
4966  tr|A0A6L3VDZ3|A0A6L3VDZ3_9BACI       None       NaN   0.00    0.666667
4967  tr|A0A6L3VET6|A0A6L3VET6_9BACI       None       NaN   0.00    0.666667
4968  tr|A0A6L3VG18|A0A6L3VG18_9BACI       None       NaN   0.00    0.666667
4969  tr|A0A6L3VGG9|A0A6L3VGG9_9BACI  TIGR00025  0.080395   0.03    1.000000
4970  tr|A0A6L3VI01|A0A6L3VI01_9BACI  TIGR00025  0.068220   0.02    1.000000

[4971 rows x 5 columns]
parsin

## Apply Pipeline is done.

Output is located in /output/apply/kmer-summary-{input file name}.csv.

Each output file has 5 columns.
* **SeqID**: ID of the sequence whose annotation we are trying to predict.  
* **Prediction**: The predicted annotatation for the sequence.  
* **Score**: The cosine similarity score between the sequence and the predicted annotation.  
* **Delta**: The difference of cosine similarity scores between the top two predicted values.  
* **Confidence**: The estimated confidence of the prediction. This is based on the global distribution. Confidence will be more accurate for annotations with more training sequences.  

